# Generador Sintético v3 - Production Grade
## Dataset con 160+ Defectos Inyectados

Este notebook genera un dataset sintético que replica la estructura real de Auditoria3:
- **27 campos de flota** (vehículos): matricula, dominio, especificaciones, tarjeta, estado
- **21 campos de dispositivos** (telemetría): GPS, batería, ignición, movimiento
- **Eventos de telemetría**: transmisiones, movimiento, recargas
- **Transacciones de combustible**: litros, precio, tarjeta
- **Ground truth**: 160+ defectos inyectados con registro completo

Reproducible mediante seed=20260816

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✓ Google Drive montado")

In [ ]:
import random
import json
from datetime import datetime, timedelta
from dataclasses import dataclass
from typing import Dict, List
import pandas as pd
from pathlib import Path

# ============================================================================
# CONFIG
# ============================================================================

@dataclass
class GenerationConfig:
    """Configuration for dataset generation"""
    num_vehicles: int = 200
    num_devices_per_vehicle: float = 0.85
    num_events_per_device: int = 50
    num_fuel_transactions: int = 500
    seed: int = 20260816
    scenario: str = "early_stage_v3"
    defect_duplicate_domain_pct: float = 0.02
    defect_duplicate_matricula_pct: float = 0.02
    defect_invalid_domain_pct: float = 0.03
    defect_missing_values_pct: float = 0.05
    defect_invalid_type_pct: float = 0.02
    defect_format_drift_pct: float = 0.04
    defect_inconsistency_pct: float = 0.04

In [ ]:
# ============================================================================
# HELPERS
# ============================================================================

def _id(min_val: int = 1, max_val: int = 65535) -> int:
    return random.randint(min_val, max_val)

def _decimal(min_val: float, max_val: float, decimals: int = 2) -> float:
    return round(random.uniform(min_val, max_val), decimals)

def _letters(length: int = 3) -> str:
    return ''.join(random.choices('ABCDEFGHIJKLMNOPQRSTUVWXYZ', k=length))

def _digits(length: int = 3) -> str:
    return ''.join(random.choices('0123456789', k=length))

def _argentine_domain() -> str:
    return _letters(2) + _digits(3) + _letters(2)

def _vin() -> str:
    return ''.join(random.choices('ABCDEFGHJKLMNPRSTUVWXYZ0123456789', k=17))

def _motor_number() -> str:
    return ''.join(random.choices('ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789', k=10))

def _date_str(start_offset_days: int = -365, end_offset_days: int = 0) -> str:
    base = datetime(2026, 9, 4)
    start = base + timedelta(days=start_offset_days)
    end = base + timedelta(days=end_offset_days)
    random_date = start + timedelta(days=random.random() * (end - start).days)
    return random_date.strftime('%Y-%m-%d')

def _datetime_str(start_offset_days: int = -30, end_offset_days: int = 0) -> str:
    base = datetime(2026, 9, 4, 12, 0, 0)
    start = base + timedelta(days=start_offset_days)
    end = base + timedelta(days=end_offset_days)
    random_dt = start + timedelta(seconds=random.random() * (end - start).total_seconds())
    return random_dt.strftime('%Y-%m-%d %H:%M:%S')

In [ ]:
# ============================================================================
# CLEAN DATA GENERATION
# ============================================================================

def generate_vehicles(num: int) -> pd.DataFrame:
    dependencias = ['JEFATURA', 'OPERACIONES', 'LOGISTICA', 'APOYO', 'DIRECCIÓN']
    tipos = ['PICK-UP', 'PICK-UP 4X4', 'AUTO', 'CAMIÓN', 'MINIBUS', 'MOTO']
    marcas = ['NISSAN', 'TOYOTA', 'FORD', 'VOLKSWAGEN', 'CHEVROLET', 'HONDA']
    colores = ['BLANCO', 'NEGRO', 'GRIS', 'AZUL', 'ROJO', 'VERDE', None]
    combustibles = ['GASOIL', 'NAFTA', 'GASOLINA', 'GNC']
    relacion = ['A', 'B', 'C', 'D', 'E', 'J', 'K']
    estados = ['En Servicio', 'Fuera de Servicio', 'Baja', 'Mantenimiento']

    data = []
    for i in range(num):
        data.append({
            'Matricula': str(_id(10000, 11999)),
            'Dominio': _argentine_domain(),
            'Dependencia': random.choice(dependencias),
            'Identificable': random.choice(['SI', 'NO']),
            'TipoVehiculo': random.choice(tipos),
            'Marca': random.choice(marcas),
            'Modelo': random.choice(['FRONTIER', 'HILUX', 'RANGER', 'F100']),
            'Color': random.choice(colores),
            'NumeroChasis': _vin() if random.random() > 0.45 else None,
            'NumeroMotor': _motor_number() if random.random() > 0.45 else None,
            'Año': random.randint(2000, 2026),
            'Procedencia': random.choice([None, 'Original', 'Secuestro']),
            'TipoCombustible': random.choice(combustibles),
            'CapacidadTanque': random.randint(40, 120),
            'RelacionConsumo': random.choice(relacion),
            'ExcepcionOdometro': random.choice(['SI', 'NO']),
            'FechaHastaExcepcionOdometro': _date_str(-180, 90) if random.random() > 0.98 else None,
            'NumeroTarjeta': f'{_digits(20)}' if random.random() > 0.05 else None,
            'NumeroContrato': random.randint(1, 7),
            'LimiteSaldo': _decimal(1000000, 10000000, 2),
            'LimiteLitros': random.randint(500, 5000),
            'RetiraDni': _id(10000000, 45000000),
            'RetiraNombre': f'{_letters(6)} {_letters(8)}' if random.random() > 0.05 else None,
            'Cupo': random.randint(20, 200),
            'Estado': random.choice(estados),
            'SubEstado': random.choice(['ACTIVO', 'INACTIVO', None]) if random.random() > 0.51 else None,
            'DireccionGral': random.choice(['JEFATURA', 'OPERACIONES']),
        })
    return pd.DataFrame(data)

def generate_devices(vehicles: pd.DataFrame) -> pd.DataFrame:
    device_types = ['Automotor', 'Motocicleta']
    modelos = ['Starlink ER-01', 'Starlink ER-02', 'Tracker X1']
    grupos = ['ACTIVOS', 'BAJAS / REM', 'MANTENIMIENTO']
    estados_trans = ['Activo', 'Apagado o falla', 'Inactivo']
    estados_ign = ['Vehículo apagado', 'Vehículo encendido']
    estados_mov = ['Detenido', 'Movimiento lento', 'Movimiento rápido']

    data = []
    device_id = 1

    for _, veh in vehicles.iterrows():
        if random.random() < 0.85:  # 85% tienen dispositivos
            num_devs = 1 if random.random() > 0.1 else 2
            for _ in range(num_devs):
                data.append({
                    'Secuencia': device_id,
                    'Tipo Dispositivo': random.choice(device_types),
                    'Grupo': random.choice(grupos),
                    'Modelo equipo': random.choice(modelos),
                    'Placa': veh['Dominio'],
                    'MSISDN': _decimal(34.0, 35.0, 10),
                    'Alias': f"{veh['Matricula']} {_id(1000, 9999)}",
                    'IMEI': _id(100000000000000, 999999999999999),
                    'Latitud': _decimal(-34.8, -29.7, 6),
                    'Longitud': _decimal(-65.5, -58.7, 6),
                    'Hora de última transmisión': _datetime_str(-7, 0),
                    'Hora de última posición válida': _datetime_str(-7, 0),
                    'Estado candado': None,
                    'Estado de transmisión': random.choice(estados_trans),
                    'Estado de ignición': random.choice(estados_ign),
                    'Estado de movimiento': random.choice(estados_mov),
                    '% Batería': random.randint(10, 100),
                    'Odómetro (Km)': _decimal(1000, 500000, 2),
                    '% Batería Externa': random.randint(0, 100),
                    'Valor Batería Externa': random.randint(0, 15000),
                    'Horómetro (hrs)': _decimal(100, 10000, 2),
                })
                device_id += 1
    return pd.DataFrame(data)

def generate_telemetry_events(devices: pd.DataFrame, events_per_device: int = 50) -> pd.DataFrame:
    data = []
    for _, device in devices.iterrows():
        for _ in range(events_per_device):
            data.append({
                'id': len(data) + 1,
                'dispositivo_id': device['Secuencia'],
                'fecha_evento': _datetime_str(-30, 0),
                'tipo_evento': random.choice(['ENCENDIDO', 'APAGADO', 'MOVIMIENTO', 'RECARGA', 'ALERTA']),
                'latitud': device['Latitud'] + _decimal(-0.1, 0.1, 6),
                'longitud': device['Longitud'] + _decimal(-0.1, 0.1, 6),
                'velocidad_kmh': _decimal(0, 120, 1),
                'odometro_km': _decimal(1000, 500000, 2),
                'bateria_pct': random.randint(10, 100),
            })
    return pd.DataFrame(data)

def generate_fuel_transactions(vehicles: pd.DataFrame, num_trans: int = 500) -> pd.DataFrame:
    data = []
    for trans_id in range(num_trans):
        veh = random.choice(vehicles.values)
        data.append({
            'id': trans_id + 1,
            'vehiculo_id': veh[0],
            'fecha': _datetime_str(-90, 0),
            'estacion': random.choice(['YPF', 'Shell', 'Axion', 'Puma']),
            'litros': _decimal(10, 100, 2),
            'precio_unitario': _decimal(1.0, 2.5, 2),
            'importe_total': _decimal(100, 250, 2),
            'numero_tarjeta': f"{_digits(20)}",
        })
    return pd.DataFrame(data)

In [ ]:
# ============================================================================
# DEFECT INJECTION
# ============================================================================

def inject_defects(vehicles: pd.DataFrame, config: GenerationConfig) -> tuple:
    gt = []
    vehicles = vehicles.copy()

    # DQ_DUP_DOMAIN
    count = int(len(vehicles) * config.defect_duplicate_domain_pct)
    indices = random.sample(range(len(vehicles)), min(count, len(vehicles) - 1))
    for idx in indices:
        src = random.choice([i for i in range(len(vehicles)) if i != idx])
        vehicles.at[idx, 'Dominio'] = vehicles.at[src, 'Dominio']
        gt.append({'id': len(gt), 'tipo': 'DQ_DUP_DOMAIN', 'entidad': 'vehiculo',
                   'registro_id': vehicles.at[idx, 'Matricula'], 'severidad': 'alta',
                   'parametros': json.dumps({'dominio': vehicles.at[idx, 'Dominio']}),
                   'descripcion': f"Dominio duplicado: {vehicles.at[idx, 'Dominio']}"})

    # DQ_DUP_VEH_ID
    count = int(len(vehicles) * config.defect_duplicate_matricula_pct)
    indices = random.sample(range(len(vehicles)), min(count, len(vehicles) - 1))
    for idx in indices:
        src = random.choice([i for i in range(len(vehicles)) if i != idx])
        vehicles.at[idx, 'Matricula'] = vehicles.at[src, 'Matricula']
        gt.append({'id': len(gt), 'tipo': 'DQ_DUP_VEH_ID', 'entidad': 'vehiculo',
                   'registro_id': vehicles.at[idx, 'Matricula'], 'severidad': 'alta',
                   'parametros': json.dumps({'matricula': vehicles.at[idx, 'Matricula']}),
                   'descripcion': f"Matrícula duplicada: {vehicles.at[idx, 'Matricula']}"})

    # DQ_INVALID_FORMAT
    count = int(len(vehicles) * config.defect_invalid_domain_pct)
    indices = random.sample(range(len(vehicles)), min(count, len(vehicles)))
    for idx in indices:
        bad = random.choice([_digits(6), _letters(6), 'XX9999XX', ''])
        vehicles.at[idx, 'Dominio'] = bad
        gt.append({'id': len(gt), 'tipo': 'DQ_INVALID_FORMAT', 'entidad': 'vehiculo',
                   'registro_id': vehicles.at[idx, 'Matricula'], 'severidad': 'media',
                   'parametros': json.dumps({'campo': 'Dominio'}),
                   'descripcion': f"Dominio en formato inválido: {bad}"})

    # DQ_MISSING_VALUE
    count = int(len(vehicles) * config.defect_missing_values_pct)
    indices = random.sample(range(len(vehicles)), min(count, len(vehicles)))
    for idx in indices:
        field = random.choice(['Marca', 'Modelo', 'TipoCombustible', 'NumeroTarjeta'])
        vehicles.at[idx, field] = None
        gt.append({'id': len(gt), 'tipo': 'DQ_MISSING_VALUE', 'entidad': 'vehiculo',
                   'registro_id': vehicles.at[idx, 'Matricula'], 'severidad': 'media',
                   'parametros': json.dumps({'campo': field}),
                   'descripcion': f"Valor faltante en {field}"})

    # DQ_INVALID_TYPE
    count = int(len(vehicles) * config.defect_invalid_type_pct)
    indices = random.sample(range(len(vehicles)), min(count, len(vehicles)))
    vehicles['Año'] = vehicles['Año'].astype('object')
    for idx in indices:
        bad = _letters(4)
        vehicles.at[idx, 'Año'] = bad
        gt.append({'id': len(gt), 'tipo': 'DQ_INVALID_TYPE', 'entidad': 'vehiculo',
                   'registro_id': vehicles.at[idx, 'Matricula'], 'severidad': 'alta',
                   'parametros': json.dumps({'campo': 'Año'}),
                   'descripcion': f"Tipo inválido en Año: {bad}"})

    # DQ_FORMAT_DRIFT
    count = int(len(vehicles) * config.defect_format_drift_pct)
    indices = random.sample(range(len(vehicles)), min(count, len(vehicles)))
    for idx in indices:
        vehicles.at[idx, 'Dominio'] = vehicles.at[idx, 'Dominio'].lower()
        gt.append({'id': len(gt), 'tipo': 'DQ_FORMAT_DRIFT', 'entidad': 'vehiculo',
                   'registro_id': vehicles.at[idx, 'Matricula'], 'severidad': 'baja',
                   'parametros': json.dumps({'campo': 'Dominio'}),
                   'descripcion': f"Formato inconsistente: {vehicles.at[idx, 'Dominio']}"})

    # DQ_INCONSISTENCY
    count = int(len(vehicles) * config.defect_inconsistency_pct)
    indices = random.sample(range(len(vehicles)), min(count, len(vehicles)))
    for idx in indices:
        vehicles.at[idx, 'Año'] = 2030
        gt.append({'id': len(gt), 'tipo': 'DQ_INCONSISTENCY', 'entidad': 'vehiculo',
                   'registro_id': vehicles.at[idx, 'Matricula'], 'severidad': 'media',
                   'parametros': json.dumps({'problema': 'year_in_future'}),
                   'descripcion': 'Año de vehículo en el futuro'})

    return vehicles, pd.DataFrame(gt)

In [ ]:
# ============================================================================
# EXECUTION
# ============================================================================

print("\n" + "="*80)
print("🚀 SYNTHETIC DATA GENERATION v3 - Production Grade")
print("="*80 + "\n")

# Configure
config = GenerationConfig(num_vehicles=200)
random.seed(config.seed)

# Generate clean data
print("📊 Generating clean data...")
vehicles = generate_vehicles(config.num_vehicles)
devices = generate_devices(vehicles)
telemetry = generate_telemetry_events(devices, config.num_events_per_device)
combustible = generate_fuel_transactions(vehicles, config.num_fuel_transactions)

print(f"  ✓ vehiculo:               {len(vehicles):6d} rows × 27 cols")
print(f"  ✓ dispositivo:            {len(devices):6d} rows × 21 cols")
print(f"  ✓ evento_telemetria:      {len(telemetry):6d} rows ×  9 cols")
print(f"  ✓ transaccion_combustible: {len(combustible):6d} rows ×  8 cols")

# Inject defects
print("\n🔴 Injecting defects...")
vehicles, ground_truth = inject_defects(vehicles, config)

print(f"  ✓ Total defects:          {len(ground_truth):6d}")
if len(ground_truth) > 0:
    print(f"\n  Defects by type:")
    for tipo, count in ground_truth['tipo'].value_counts().items():
        print(f"    - {tipo:30s}: {count:3d}")

# Execution registry
ejecucion = pd.DataFrame([{
    'timestamp': datetime.now().isoformat(),
    'seed': config.seed,
    'scenario': config.scenario,
    'num_vehicles': len(vehicles),
    'num_defects': len(ground_truth),
}])

print(f"\n{'='*80}")
print(f"✅ DATASET GENERATED SUCCESSFULLY")
print(f"{'='*80}\n")

In [ ]:
# ============================================================================
# EXPORT TO GOOGLE DRIVE
# ============================================================================

output_dir = Path('/content/drive/MyDrive/Integrador/datasets/early_stage_v3')
output_dir.mkdir(parents=True, exist_ok=True)

print(f"💾 Exporting to Google Drive: {output_dir}\n")

# Export CSVs
tables = {
    'vehiculo': vehicles,
    'dispositivo': devices,
    'evento_telemetria': telemetry,
    'transaccion_combustible': combustible,
    'ground_truth': ground_truth,
    'ejecucion_dataset': ejecucion,
}

for name, df in tables.items():
    path = output_dir / f"{name}.csv"
    df.to_csv(path, index=False)
    print(f"  ✓ {name:30s} → {path.name}")

# Export manifest
manifest = {
    'seed': config.seed,
    'scenario': config.scenario,
    'generated_at': datetime.now().isoformat(),
    'tables': {
        name: {'rows': len(df), 'cols': len(df.columns)}
        for name, df in tables.items()
    },
    'defects': {
        tipo: int(count)
        for tipo, count in ground_truth['tipo'].value_counts().items()
    },
}

manifest_path = output_dir / 'manifest.json'
with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

print(f"\n  ✓ manifest.json exported")
print(f"\n✅ All files saved to Google Drive")